In [ ]:
# ---------- inline code bundle (written verbatim; no dataset) ----------
import os, pathlib
ROOT = '/kaggle/working/code'
p = pathlib.Path(ROOT, 'experiments/exp47_common.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""E-47 common: численная кинематика листа L23 (общое ядро для exp47a/b/c/d).\n\nМатематика (полный вывод и верификация — docs/leaf_dynamics.md §2,§4):\n\n    J(h) = r(h) * M (I - rho * u u^T),   u = h/||h||,  rho = ||h||^2/(||h||^2 + D*eps),\n    M    = W_head[U128] * diag(1 + w_norm),          # Qwen RMSNorm: (1+w), НЕ w\n    r(h) = rsqrt(mean(h^2) + eps)                     # скаляр -> на V не влияет\n\nВся зависимость V_16(h) от позиции — ЧЕРЕЗ ОДНУ u (ранг-1 наклон). Отсюда:\n  * V инвариантна к масштабу h;\n  * delta J от поворота u на du ⟂ u: -rho*M(du u^T + u du^T) — ранг 2;\n  * скорость листа s_t пропорциональна скорости потока a_t = angle(u_t,u_{t+1})\n    с резольвентным множителем 1/(sigma_i^2 - sigma_j^2) (док. gap16 мал ⇒ рама жёсткая).\n\nЧисленная гигиена (урок верификации 2026-11-05): собственные векторы Gram берём от\nJJ^T размера [K,K] (K=128), а не J^T J [D,D]: второе удваивает обусловленность на блоке D\nи в fp32 даёт ложный пол ~1e-3 рад на масштабе s_t. Путь: eigh(JJ^T) -> U, sigma^2;\nправые векторы V = J^T U[:,top] / sigma. Пол метода в fp64: 1.4e-6 рад (проверено).\n\nВсе функции чистые, torch-only (scipy.linalg.schur только для углов Шюра).\n"""\nfrom __future__ import annotations\n\nimport math\n\nimport torch\n\n\n# ---------------------------------------------------------------- Jacobian / V\n\ndef head_M(head_W: torch.Tensor, g_norm: torch.Tensor) -> torch.Tensor:\n    """M = W_head[U128] diag(g), g = 1+w_norm. fp64 — для кинематики малых углов."""\n    return (head_W.double() * g_norm.double().unsqueeze(0)).contiguous()\n\n\ndef J_analytic(A: torch.Tensor, h: torch.Tensor, eps: float) -> tuple[torch.Tensor, float, torch.Tensor]:\n    """J(h) без скаляра r (на V не влияет): J = A (I - rho u u^T) = A - rho (A u) u^T.\n\n    Возвращает (J [K,D], rho, u [D]). A = M (fp64). Одна внешняя произведение — O(K D).\n    """\n    hh = float(h @ h)\n    D = h.numel()\n    rho = hh / (hh + D * eps)\n    u = h / math.sqrt(hh)\n    J = A - rho * torch.outer(A @ u, u)\n    return J, rho, u\n\n\ndef V16_from_J(J: torch.Tensor, r: int = 16):\n    """Топ-r правых сингулярных подпространств через Gram JJ^T [K,K] (fp64).\n\n    V = J^T U_k / sigma_k (U_k — собственные векторы JJ^T с бОльшими lmbda=sigma^2).\n    Возвращает (V [D,r], sigma [K] — все, по порядку убывания).\n    """\n    G = J @ J.T\n    G = 0.5 * (G + G.T)\n    lam, U = torch.linalg.eigh(G)          # по возрастанию\n    idx = torch.argsort(lam, descending=True)\n    lam, U = lam[idx], U[:, idx]\n    sig = lam.clamp_min(1e-30).sqrt()\n    V = (J.T @ U[:, :r]) / sig[:r]\n    return V.contiguous(), sig\n\n\ndef procrustes_R(Va: torch.Tensor, Vb: torch.Tensor) -> torch.Tensor:\n    """Ортогональный транспорт рамы Va->Vb: R = polar(Va^T Vb) = U W^T (SVD 16x16).\n\n    R определён корректно, когда столбцы — УПОРЯДОЧЕННЫЕ базисы (наши V из eigh — да).\n    Знаки столбцов SVD-базисов тут не мешают: Va->Va даёт точную I только при совпадении\n    знаков; поэтому дальше используем только инварианты R (углы Шюра), которые от знака\n    базиса НЕ зависят (подправка знаком вписана в сам полярный множитель).\n    """\n    U, _, Wt = torch.linalg.svd(Va.T @ Vb, full_matrices=False)\n    return U @ Wt\n\n\n# ---------------------------------------------------------------- инварианты R\n\ndef schur_angles(R: torch.Tensor):\n    """Главная часть log(R) для R in SO(16): углы 2x2-блоков вещественного Шюра.\n\n    Вещественный Шур R = Q T Q^T: диагональные 1x1 блоки = ±1 (угол 0 или pi),\n    2x2 блоки [[a,b],[c,d]]bc<0: собственные a ± i sqrt(|bc|) => theta = atan2(|b|,a)\n    (берём |theta| в [0,pi)). Возвращает (theta sorted desc [m], Q — векторы Шюра,\n    bsize — размеры блоков): Q[:, k] — ИНВАРИАНТНОЕ подпространство блока k\n    (для 2-блока — плоскость вращения с наибольшим |theta|, если блоки отсортированы).\n    """\n    import numpy as np\n    from scipy.linalg import schur\n\n    T, Q = schur(R.detach().cpu().numpy(), output="real")\n    n = T.shape[0]\n    thetas: list[float] = []\n    bsize: list[int] = []\n    i = 0\n    while i < n:\n        if i + 1 < n and abs(T[i + 1, i]) > 1e-10:\n            # собственный блок Шюра: [[a,b],[c,d]], bc<0, собств. = (a+d)/2 ± i*sqrt|bc|\n            re = 0.5 * (float(T[i, i]) + float(T[i + 1, i + 1]))\n            im = math.sqrt(abs(float(T[i, i + 1]) * float(T[i + 1, i])))\n            th = math.atan2(im, re) % math.pi\n            thetas.append(th)\n            bsize.append(2)\n            i += 2\n        else:\n            th = 0.0 if float(T[i, i]) > 0 else math.pi\n            thetas.append(th)\n            bsize.append(1)\n            i += 1\n    order = sorted(range(len(thetas)), key=lambda k: -thetas[k])\n    # переставляем векторы Шюра блоками в порядке убывания угла\n    cols: list[int] = []\n    start: list[int] = []\n    s = 0\n    for bs in bsize:\n        start.append(s)\n        s += bs\n    for k in order:\n        cols.extend(range(start[k], start[k] + bsize[k]))\n    Q2 = torch.from_numpy(Q[:, cols])\n    return torch.tensor([thetas[k] for k in order], dtype=torch.float64), Q2, [bsize[k] for k in order]\n\n\ndef frame_kinematics(Va: torch.Tensor, Vb: torch.Tensor):\n    """Один шаг кинематики ЛИСТА (подпространства) — базис-инвариантные величины.\n\n    ⚠ УРОК (найден первым прогоном P0 2026-11-05): углы Шюра одного шага Procrustes\n    R = polar(Va^T Vb) НЕ инвариантны к внутренним базисам рам: Va->Va@Q1, Vb->Vb@Q2\n    дают R -> Q1^T R Q2, собственные углы при этом меняются. Внутренний базис eigh/SVD\n    при плотном кластере сингулярных чисел выбирается произвольно — одношаговый R мерил\n    «рассогласование базисов» (s~π, e~π/2 на ровных данных). Инвариантен ТОЛЬКО цикличный\n    произведение (голономия κ — потому E-28 и корректен). Для одного шага — ГЛАВНЫЕ УГЛЫ\n    подпространств (Грассманиан):\n        Va^T Vb = X diag(cos Θ) Y^T ;  s = ||Θ||_2,  p = Θ_max/s,\n        собственная 2-плоскость в D-координатах: span{Va X_1, Vb Y_1} (инвариантна).\n    В меловом limite Θ = углы вращения, так что семантика «скорость/планарность» сохранена.\n    Возвращает (s, p, P [D,2], Θ [r] desc).\n    """\n    M = Va.T @ Vb\n    X, c, Yh = torch.linalg.svd(M, full_matrices=False)\n    Y = Yh.T\n    Th = torch.acos(c.clamp(-1.0 + 1e-12, 1.0 - 1e-12))\n    # svd отдаёт cos Θ по убыванию ⇒ Θ по ВОЗРАСТАНИЮ; доминирующий — последний\n    order = torch.argsort(Th, descending=True)\n    Th = Th[order]\n    i0 = int(order[0])\n    s = float(torch.sqrt((Th * Th).sum()))\n    p = float(Th[0] / s) if s > 1e-12 else float("nan")\n    P = torch.column_stack([Va @ X[:, i0], Vb @ Y[:, i0]]).double().contiguous()\n    return s, p, P, Th\n\n\ndef align_to_u_plane(P: torch.Tensor, u0: torch.Tensor, u1: torch.Tensor) -> float:\n    """Худший главный угол (рад) между 2-плоскостью P и span{u0,u1}; nan если span вырожден.\n\n    Для 2-плоскости в D>4 корректнее брать ПЛОЩАДНУЮ согласованность: здесь худший\n    главный угол — если P и span расходятся хотя бы в одном направлении, угол велик.\n    cos главного угла = сингулярные числа P^T Q_U; acos(min) — угол самого «непокрытого»\n    направления: 0 = плоскости совпадают.\n    """\n    B = torch.column_stack([u0, u1])\n    svB = torch.linalg.svdvals(B)\n    if float(svB.min()) < 1e-8 * float(svB.max()):\n        return float("nan")            # u0 || u1: шага нет, выравнивание не определено\n    Qb = torch.linalg.qr(B)[0]\n    svP = torch.linalg.svdvals(P)\n    if float(svP.min()) < 1e-6 * float(svP.max()):\n        return float("nan")            # Θ мал: «плоскость» выродилась в прямую, QR бы соврал\n    Qp = torch.linalg.qr(P)[0]         # P — два ПРИНЦИПАЛЬНЫХ вектора (угол между ними = Θ),\n                                       # не ортонормированы; без QR svd(P^T Q) врал (>1)\n    sv = torch.linalg.svdvals(Qp.T @ Qb)\n    return math.acos(float(sv.min().clamp(max=1.0 - 1e-12)))\n\n\ndef Q_orth(X: torch.Tensor) -> torch.Tensor:  # noqa: N802 (имя из формул)\n    return torch.linalg.qr(X)[0]\n\n\n# ---------------------------------------------------------------- скалярные ряды\n\ndef u_series(hs: torch.Tensor) -> torch.Tensor:\n    """По [T,D] -> единичные u_t (fp64)."""\n    H = hs.double()\n    return H / H.norm(dim=-1, keepdim=True)\n\n\ndef angle_series(u: torch.Tensor) -> torch.Tensor:\n    """a_t = angle(u_t, u_{t+1}) [T-1]."""\n    c = (u[:-1] * u[1:]).sum(-1).clamp(max=1.0 - 1e-12)\n    return torch.acos(c)\n\n\ndef omega_geodesic(u: torch.Tensor) -> torch.Tensor:\n    """Геодезическая кривизна [M-3]: |A01 + A12 - A02| по тройкам (T-2)."""\n    def A(x, y):\n        return torch.acos((x * y).sum(-1).clamp(max=1.0 - 1e-12))\n    return (A(u[:-2], u[1:-1]) + A(u[1:-1], u[2:]) - A(u[:-2], u[2:])).abs()\n\n\ndef band_power(x: torch.Tensor, lo: int = 4, hi: int = 32) -> float:\n    """EXCESS спектральной плотности в полосе периодов [lo,hi] против вне-полосы.\n\n    =1 белый шум; ≫1 — периодическая компонента на этих периодах (детектор орбиты).\n    Доля-в-полосе не годится: полоса покрывает почти весь спектр короткого ряда, и даже\n    чистая синусоида даёт ≈1. Поэтому делим среднюю плотность В полосе на среднюю ВНЕ\n    (пол с 1e-6 от общей плотности — защита от деления на ноль у чистой синусоиды).\n    Полоса по умолчанию 4..32: ниже 4 — токсично (токенизированные повторы букв),\n    выше 32 — ловим тренды вместо орбит."""\n    x = x - x.mean()\n    if x.numel() < 16 or float(x.pow(2).sum()) < 1e-20:\n        return float("nan")\n    f = torch.fft.rfft(x.double())\n    pw = (f.real ** 2 + f.imag ** 2)[1:]        # bin k <-> период N/k, DC выброшен\n    n = x.numel()\n    k_lo, k_hi = max(1, n // hi), min(pw.numel() - 1, n // lo)\n    mask = torch.zeros(pw.numel(), dtype=torch.bool)\n    mask[k_lo:k_hi + 1] = True\n    d_in = float(pw[mask].mean())\n    d_out = float(pw[~mask].mean())\n    d_tot = float(pw.mean())\n    return d_in / max(d_out, 1e-6 * d_tot)\n\n\ndef autocorr_peak(x: torch.Tensor, p: int, base_slice=slice(5, None)) -> float:\n    """ACF(x)[p] относительно медианы |ACF| хвоста — пик/база (детектор периода p).\n\n    ACF нормирована на полную энергию (смещённая оценка — ряд короткий, monotone затухание\n    вычитается базой медианы хвоста)."""\n    xf = (x - x.mean()).double()\n    if xf.numel() < 3 * p:\n        return float("nan")\n    n = xf.numel()\n    e0 = float((xf * xf).sum())\n    if e0 < 1e-20:\n        return float("nan")\n    acf = torch.tensor([float((xf[: n - k] * xf[k:]).sum() / e0)\n                        for k in range(1, min(n - 1, 3 * p + 1))])\n    base = acf[base_slice].abs().median()\n    return float(acf[p - 1] / base) if base == base and float(base) > 1e-9 else float("nan")\n\n\n# ---------------------------------------------------------------- детектор петель\n\ndef find_loops(tokens: list[int], max_p: int = 32, min_reps: int = 3):\n    """Детерминированный детектор повторов (пререгистрация docs/leaf_dynamics.md §5.2).\n\n    Эпизод: позиция j и ФУНДАМЕНТАЛЬНЫЙ период p (минимальный из подходящих), при котором\n    блок tokens[j:j+p] повторяется k>=min_reps раз подряд вперёд ( maximal\'ный прогон).\n    onset = j + (min_reps-1)*p — начало ТРЕТЬЕГО повтора («петля объявила себя»). Эпизоды\n    с одинаковым onset склеиваются; в выход кладётся максимальный прогон по каждому onset.\n    Возвращает [(onset, start=j, end, p, k)].\n    """\n    n = len(tokens)\n    raw: dict[int, tuple[int, int, int, int]] = {}\n    for j in range(n - 2):\n        for p in range(2, max_p + 1):\n            block = tokens[j: j + p]\n            if len(block) < p:\n                break\n            k = 1\n            e = j + p\n            while e + p <= n and tokens[e: e + p] == block:\n                k += 1\n                e += p\n            if k >= min_reps and p * (k - 1) >= 6:   # подавление тривиальных «aa», «aab aab»\n                cand = (j, e, p, k)\n                prev = raw.get(j)\n                if prev is None or p * k > prev[2] * prev[3]:\n                    raw[j] = cand\n    out = sorted((j + (min_reps - 1) * cand[2], j, cand[1], cand[2], cand[3])\n                 for j, cand in raw.items())\n    # склейка: эпизод, начинающийся внутри более длинного с тем же периодом, отбрасывается\n    merged: list[tuple[int, int, int, int, int]] = []\n    for ep in sorted(out, key=lambda x: (x[1], -x[4] * x[3])):\n        if merged and merged[-1][1] <= ep[1] < merged[-1][2] and merged[-1][3] == ep[3]:\n            continue\n        merged.append(ep)\n    return merged\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'experiments/exp47b_gen_kinematics.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('#!/usr/bin/env python\n"""E-47b (P1): кинематика листа L23 при СВОБОДНОЙ ГЕНЕРАЦИИ (Kaggle T4 / локально).\n\nПостановка/критерии (K1.1–K1.4, пререгистрированы) — docs/leaf_dynamics.md §5.2,§6;\nчисла P0 и прогноз (насыщение s=0.383·tanh(a/0.715), выход с полки на мелком шаге) — §8.1.\n\nПротокол:\n  * 4 домена x n_docs документов; промпт = первые prompt_len токенов калибровочного пула\n    (те же датасеты, что строили кэш exp27b; точное совпадение документов НЕ гарантируется\n    потоком датасета — пишем sha256 промптов в артефакт);\n  * greedy-генерация max_new шагов с DynamicCache; на каждый шаг берём h = hidden_states[n_layers]\n    (выход L23 = вход финальной RMSNorm; НЕ last_hidden_state — тот уже post-norm);\n  * кинематика из exp47_common: a_t, s_t (главные углы V16), p_t, g_t (доля движения u,\n    видимая листу: ||Pi_{span(Va∪Vb)} du|| / ||du||), Omega_t; тривиальные фичи-конкуренты:\n    энтропия logits, top-1 prob, флаг «токен=предыдущий»;\n  * ПЕТЛИ: find_loops по сгенерированным токенам (onset = начало 3-го повтора);\n  * СОХРАНЯЕМ СЫРЬЁ (npz): фичи per step, V16 fp16 [T,16,D], u fp16 — весь P2/P3-анализ\n    пересобирается офлайн без перепрогона.\n\nБазис U128 — КОНСТАНТА из кэша exp27b (источник истины всех опубликованных чисел;\nисточник: exp27b_jcache_L23.pt, pool_fp wiki 0cc16ec344e2 / math e71cc282c85d / code\n209e83b1898b / chat 2afac288344a), с двумя самопроверками на лету:\n  (i)  маршрут: H_head @ W_lm^T vs model(...).logits < 1e-3 (как exp45b) — падает громко;\n  (ii) частотный базис, пересобранный с пула этим же датасетом — пересечение в % (дескриптив;\n       поток датасета мог сдвинуться — на выводы не влияет, U128 фиксирован константой).\n\nKaggle: сигнатура main(model=..., device=..., results_dir=...) — драйвер билда подставляет\nлокальный путь модели/устройство/каталог. Локально: аргументы CLI.\n\nЗапуск (локально, CPU медленно): ../.venv/bin/python exp47b_gen_kinematics.py --n-docs 2\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nimport sys\nimport time\n\nimport numpy as np\nimport torch\n\n_HERE = os.path.dirname(os.path.abspath(__file__))\nif _HERE not in sys.path:\n    sys.path.insert(0, _HERE)\n_ROOT = os.path.dirname(_HERE)          # Kaggle: бандл примонтирован как <root>/{prism,experiments}\nif _ROOT not in sys.path:\n    sys.path.insert(0, _ROOT)\n\nimport exp47_common as C   # noqa: E402\nfrom prism.calibration import build_global_token_set, load_calibration_data  # noqa: E402\n\nRNG_SEED = 47\n# U128 — частотный базис exp27b (константа-источник; см. докстринг)\nU128 = [0, 3, 4, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25,\n        26, 30, 32, 82, 87, 198, 220, 264, 271, 279, 303, 310, 314, 318, 321, 328, 348,\n        351, 353, 357, 364, 369, 383, 393, 411, 413, 421, 423, 424, 430, 440, 449, 458,\n        466, 471, 494, 498, 506, 513, 524, 539, 579, 599, 615, 628, 635, 641, 653, 660,\n        668, 694, 695, 716, 725, 735, 748, 777, 791, 799, 821, 864, 948, 984, 1000, 1040,\n        1066, 1076, 1088, 1092, 1116, 1132, 1172, 1220, 1238, 1317, 1330, 1332, 1345,\n        1358, 1381, 1472, 1518, 1521, 1532, 1602, 1608, 1680, 1754, 1831, 1973, 2107,\n        2228, 2394, 2397, 2424, 2574, 2582, 2785, 2957, 2972, 3018, 3067, 4370, 4558,\n        56014, 248044, 248046]\nPOOL_FP_EXPECT = {"wiki": "0cc16ec344e2", "math": "e71cc282c85d",\n                  "code": "209e83b1898b", "chat": "2afac288344a"}\n\n\ndef _sha(x) -> str:\n    return hashlib.sha256(str(x).encode()).hexdigest()[:12]\n\n\ndef _step(V_prev, u_prev, u_cur, A, eps, h_cur):\n    """Фичи перехода: V/u текущего шага + (a,s,p,g); при V_prev=None (первый шаг)\n    кинематика не определена — nan, но V/u считаем."""\n    Vb, _ = C.V16_from_J(C.J_analytic(A, h_cur, eps)[0])\n    if V_prev is None:\n        return float("nan"), float("nan"), float("nan"), float("nan"), Vb\n    Va = V_prev\n    s, p, _P, _Th = C.frame_kinematics(Va, Vb)\n    a = float(torch.acos(((u_prev * u_cur).sum()).clamp(max=1 - 1e-12)))\n    du = u_cur - (u_prev @ u_cur) * u_prev\n    ndu = float(du.norm())\n    g = float("nan")\n    if ndu > 1e-9:\n        q = torch.linalg.qr(torch.cat([Va, Vb], dim=1))[0]\n        g = float((q.T @ (du / ndu)).norm())\n    return a, s, p, g, Vb\n\n\ndef main(model="/data/Qwen/Qwen3.5-0.8B", n_docs=8, max_new=384, prompt_len=64,\n         device="cpu", results_dir="results", do_verify_basis=True,\n         rtn_bits=None, rtn_group=128, dtype="float32", load_in_8bit=False):\n    torch.set_num_threads(4)\n    torch.manual_seed(RNG_SEED)\n    dev = torch.device(device)\n    from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache  # type: ignore\n\n    t0 = time.time()\n    kw = {}\n    if load_in_8bit:      # честный PTQ (bnb int8): trunk квант, embed/head точные\n        from transformers import BitsAndBytesConfig\n        kw = dict(quantization_config=BitsAndBytesConfig(load_in_8bit=True),\n                  device_map="cuda:0")\n    m = AutoModelForCausalLM.from_pretrained(model, dtype=getattr(torch, dtype),\n                                             attn_implementation="eager", **kw)\n    m.eval()\n    if not load_in_8bit:\n        m.to(dev)\n    if rtn_bits:      # full-model RTN (тест A/B честный: меняется и динамика u).\n        lev = 2 ** (rtn_bits - 1) - 1\n        nq = 0\n        for mod in m.modules():\n            if isinstance(mod, torch.nn.Linear):\n                w = mod.weight.data\n                wf = w.reshape(w.shape[0], -1, rtn_group).double()\n                sc = wf.abs().amax(-1, keepdim=True).clamp(min=1e-12) / lev\n                mod.weight.data = ((wf / sc).round().clamp(-lev, lev) * sc).reshape_as(w).to(w.dtype)\n                nq += 1\n        print(f"[rtn] int{rtn_bits} per-group({rtn_group}) деQuantized Linear-слоёв: {nq}", flush=True)\n    tok = AutoTokenizer.from_pretrained(model)\n    n_layers = int(m.config.num_hidden_layers)\n    norm = m.model.norm\n    eps = float(getattr(norm, "variance_epsilon",\n                        getattr(norm, "eps", m.config.rms_norm_eps)))\n    wn = norm.weight.detach().float()\n    # эмпирика семантики RMSNorm: gemma-style g=1+w против голого g=w (Qwen2/3)\n    with torch.no_grad():\n        prb = torch.randint(1, 1000, (1, 8), device=dev)\n        pre_c = []\n        hk = norm.register_forward_hook(lambda _m, inp, _o: pre_c.append(inp[0].detach()))\n        raw = m.model(input_ids=prb).last_hidden_state[0, -1].detach().float().cpu()\n        hk.remove()\n        pre = pre_c[0][0, -1].float()\n        r = (pre.pow(2).mean(-1, keepdim=True) + eps).rsqrt()\n        e_w = float((wn * r * pre - raw).abs().max())\n        e_1w = float(((1.0 + wn) * r * pre - raw).abs().max())\n        g_mode = "w" if e_w <= e_1w else "1p_w"\n        assert min(e_w, e_1w) < 1e-3, f"norm не закрыт ни одной g: w={e_w:.1e} 1+w={e_1w:.1e}"\n        print(f"[norm] g-режим: {g_mode} (err w={e_w:.1e}, 1+w={e_1w:.1e})", flush=True)\n    g_n = (wn if g_mode == "w" else 1.0 + wn)\n    W = m.lm_head.weight.detach()\n    U = torch.tensor(U128, device=dev)\n    A = C.head_M(W[U].cpu().float(), g_n.cpu()).to(dev)      # [128,D] fp64 на устройстве\n    tied = m.lm_head.weight.data_ptr() == m.get_input_embeddings().weight.data_ptr()\n\n    # маршрут: H_head @ W_lm[U]^T == logits[U]\n    with torch.no_grad():\n        probe = torch.randint(1, 1000, (1, 8), device=dev)\n        hs = m.model(input_ids=probe).last_hidden_state[0, -1].float()\n        rt = float((hs @ W[U].T.float().cpu().to(dev)\n                    - m(input_ids=probe).logits[0, -1, U].float()).abs().max())\n    print(f"[route] max|H@W_U^T - logits_U| = {rt:.2e}; tied={tied}", flush=True)\n    assert rt < 1e-3, f"маршрут логитов не сошёлся: {rt}"\n\n    docs_out, feats_npz = [], {}\n    basis_overlap = None\n    for dom in ("wiki", "math", "code", "chat"):\n        pool = load_calibration_data(model, max_length=256, min_length=64,\n                                     n_samples=n_docs, domain=dom)\n        assert len(pool) >= n_docs\n        if do_verify_basis and dom == "wiki":       # дорого: один домен\n            fresh = build_global_token_set(m, pool[:n_docs], K=len(U128))\n            basis_overlap = len(set(fresh) & set(U128)) / len(U128)\n            print(f"[basis] overlap fresh-vs-cache U128: {basis_overlap:.2%}", flush=True)\n        for si in range(n_docs):\n            ids = pool[si]["input_ids"][:, :prompt_len].to(dev)\n            prompt_sha = _sha(pool[si]["input_ids"][0, :prompt_len].tolist())\n            # префиксный проход: h_23 промпта (последняя позиция) + cache\n            with torch.no_grad():\n                cache = DynamicCache(config=m.config)\n                out = m(input_ids=ids, past_key_values=cache, use_cache=True,\n                        output_hidden_states=True)\n            h = out.hidden_states[n_layers][0, -1].double().to(dev)   # h_23 (pre-final-norm)\n            logits = out.logits[0, -1]\n            feats = {"a": [], "s": [], "p": [], "g": [], "ent": [], "top1": [],\n                     "eq_prev": [], "hnorm": []}\n            u = h / h.norm()\n            V_prev = u_prev = None\n            # Кадр при промптной позиции — отдельно от V_prev (NaN первого шага не трогаем).\n            V_at_cur = C.V16_from_J(C.J_analytic(A, h, eps)[0])[0]\n            tokens = ids[0].tolist()\n            V_stack, u_stack = [], []\n            for step in range(max_new):\n                p = torch.softmax(logits.float(), dim=-1)\n                ent = float(-(p.clamp_min(1e-12) * p.clamp_min(1e-12).log()).sum())\n                nxt = int(torch.argmax(logits))\n                feats["ent"].append(ent)\n                feats["top1"].append(float(p.max()))\n                feats["eq_prev"].append(int(nxt == tokens[-1]))\n                feats["hnorm"].append(float(h.norm()))\n                u_stack.append(u.float().cpu().numpy())\n                tokens.append(nxt)\n                with torch.no_grad():\n                    out = m(input_ids=torch.tensor([[nxt]], device=dev),\n                            past_key_values=cache, use_cache=True,\n                            output_hidden_states=True)\n                h_new = out.hidden_states[n_layers][0, -1].double().to(dev)\n                logits = out.logits[0, -1]\n                u_new = h_new / h_new.norm()\n                a, s, pl, gg, V_cur = _step(V_prev, u, u_new, A, eps, h_new)\n                feats["a"].append(a); feats["s"].append(s)\n                feats["p"].append(pl); feats["g"].append(gg)\n                V_prev, u_prev = V_cur, u\n                u = u_new\n                # Аудит E46+ (A6): писался V_cur = кадр СЛЕДУЮЩЕЙ позиции, то есть\n                # сохранённые u[i] и V[i] были сдвинуты на шаг. Теперь оба на позиции i.\n                V_stack.append(V_at_cur.float().cpu().numpy())\n                V_at_cur = V_cur\n                h = h_new\n            gen_tokens = tokens[prompt_len:]\n            loops = C.find_loops(gen_tokens)\n            aa = np.array(feats["a"], dtype=float)\n            ss = np.array(feats["s"], dtype=float)\n            pp = np.array(feats["p"], dtype=float)\n            # посегментная сводка (in-loop по ДЕТЕРМИНИРОВАННОМУ детектору; вне — всё остальное)\n            in_mask = np.zeros(max_new, dtype=bool)\n            for lp in loops:\n                on0 = min(lp[0], max_new - 1)\n                in_mask[on0: min(lp[2], max_new)] = True\n            acf = [C.autocorr_peak(torch.from_numpy(\n                       ss[min(lp[0], len(ss)-1):min(lp[2], len(ss))] - np.nanmean(ss)), lp[3])\n                   for lp in loops]\n            r_seg = float("nan")\n            if in_mask.any() and (~in_mask).sum() > 20:\n                x, y = ss[in_mask], aa[in_mask]\n                ok = ~(np.isnan(x) | np.isnan(y))\n                if ok.sum() > 5:\n                    x = x - x.mean(); y = y - y.mean()\n                    d = math.sqrt(float((x * x).sum() * (y * y).sum()))\n                    r_seg = float((x * y).sum() / d) if d > 0 else float("nan")\n            docs_out.append({\n                "dom": dom, "si": si, "prompt_sha": prompt_sha,\n                "n_loops": len(loops),\n                "loops": [list(lp) for lp in loops],\n                "a_med": float(np.nanmedian(aa)), "s_med": float(np.nanmedian(ss)),\n                "p_med": float(np.nanmedian(pp)),\n                "g_med": float(np.nanmedian(feats["g"])),\n                "excess_a": C.band_power(torch.from_numpy(np.nan_to_num(aa))),\n                "excess_s": C.band_power(torch.from_numpy(np.nan_to_num(ss))),\n                "in_loop": {"a": float(np.nanmedian(aa[in_mask])) if in_mask.any() else None,\n                            "s": float(np.nanmedian(ss[in_mask])) if in_mask.any() else None,\n                            "p": float(np.nanmedian(pp[in_mask])) if in_mask.any() else None,\n                            "acf_peak_by_loop": [None if x != x else float(x) for x in acf],\n                            "r_s_a_in": r_seg if r_seg == r_seg else None},\n                "trivial": {"ent_med": float(np.median(feats["ent"])),\n                            "top1_med": float(np.median(feats["top1"])),\n                            "eq_prev_rate": float(np.mean(feats["eq_prev"]))},\n                "gen_sha": _sha(gen_tokens),\n            })\n            feats_npz[f"{dom}_{si}"] = {k: np.array(v, dtype=np.float32)\n                                        for k, v in feats.items()}\n            feats_npz[f"{dom}_{si}_V"] = np.stack(V_stack).astype(np.float16)\n            feats_npz[f"{dom}_{si}_u"] = np.stack(u_stack).astype(np.float16)\n            d0 = docs_out[-1]\n            print(f"{dom} s{si}: loops={d0[\'n_loops\']} a={d0[\'a_med\']:.3f} s={d0[\'s_med\']:.3f} "\n                  f"p={d0[\'p_med\']:.3f} g={d0[\'g_med\']:.3f} ent={d0[\'trivial\'][\'ent_med\']:.2f} "\n                  f"[{time.time()-t0:.0f}s]", flush=True)\n        del pool\n\n    os.makedirs(results_dir, exist_ok=True)\n    out = {"params": {"protocol": "docs/leaf_dynamics.md §5.2 (P1)", "model": str(model),\n                      "n_docs": n_docs, "max_new": max_new, "prompt_len": prompt_len,\n                      "decoding": "greedy", "seed": RNG_SEED,\n                      "U128_source": "exp27b cache constant (pool_fp см. в докстринге)",\n                      "basis_overlap_fresh": basis_overlap,\n                      "route_check": rt, "tied": bool(tied)},\n           "docs": docs_out}\n    with open(os.path.join(results_dir, "exp47b_gen_kinematics.json"), "w") as f:\n        json.dump(out, f, ensure_ascii=False, allow_nan=False)\n    feats_npz["g_norm"] = g_n.cpu().numpy().astype(np.float32)   # источник истины для e52a\n    feats_npz["eps"] = np.float64(eps)\n    np.savez_compressed(os.path.join(results_dir, "exp47b_gen_kinematics.npz"), **feats_npz)\n    print(f"saved. total {time.time()-t0:.0f}s", flush=True)\n    return out\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--model", default="/data/Qwen/Qwen3.5-0.8B")\n    ap.add_argument("--n-docs", type=int, default=8)\n    ap.add_argument("--max-new", type=int, default=384)\n    ap.add_argument("--prompt-len", type=int, default=64)\n    ap.add_argument("--device", default="cpu")\n    ap.add_argument("--rtn-bits", type=int, default=None,\n                    help="full-model RTN: симметрично per-group квантовать ВСЕ Linear (dequant in-place)")\n    ap.add_argument("--rtn-group", type=int, default=128)\n    ap.add_argument("--results-dir", default=os.path.join(_HERE, "..", "results"))\n    a = ap.parse_args()\n    main(model=a.model, n_docs=a.n_docs, max_new=a.max_new, prompt_len=a.prompt_len,\n         device=a.device, results_dir=a.results_dir,\n         rtn_bits=a.rtn_bits, rtn_group=a.rtn_group)\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'experiments/exp49_leaf_info.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('#!/usr/bin/env python\n"""E-49: НЕСЁТ ЛИ ЛИСТ ИНФОРМАЦИЮ О ТРАЕКТОРИИ — размороженный протокол с контролями.\n\nКонтекст: тождество J и «облако» V-рам воспроизводятся на двух архитектурах (E-48), но\nпопарные эвклидовы метрики r=16 оказались null-эквивалентны (§10). Проверяем три гипотезы\nо том, что МЫ МЕРИЛИ НЕ ТАК, каждая — несколькими методами:\n\n  H1 «не тот top»     — r ∈ {1,2,4} (верхние моды, σ1/σ2≈2.3) vs 16 (вырожденный хвост).\n  H2 «не та гранулярн.»— сглаживание h по окну W ∈ {1,4,16} открывает медленный канал.\n  H3 «не та метрика»  — углы в pullback-метрике ридаута G = AᵀA (M-метрика, G-ортонорм.\n                        базисы через L=chol(G)).\n\nМетоды на каждом (r, W, метрика):\n  M1 лаг-кривая s(lag), lag ∈ {1,2,4,8,16,32} (full-length серии, без прореживания);\n  M2 когерентность τ = [s(t,t+1)+s(t+1,t+2)−s(t,t+2)]/s(t,t+1)  (0 = связное движение,\n     1 = белое; главный дискриминант, работает даже когда лаг-кривая насыщена);\n  M3 голономия окна L=8 — НЕ ЗАПУСКАЛАСЬ в production-сетке (функция holonomy()\n     есть, не вызывалась; исправлено по ревью E49 §1.2 — заявка снята);\n  M4 v1-угол со знакомым транспортом (r=1 чувствителен к знаку — фиксируем трассировкой).\n\nКОНТРОЛИ (падение этапа, если не различились):\n  C-A white u (независимые направления через тот же конвейер): τ≈1, лаг flat;\n  C-B coherent u (шаги δ=0.02 по геодезическим, медленно поворачивающаяся плоскость):\n      τ<0.5, лаг-кривая растёт.\n\nЗапуск: --stage 1 (только контроли, быстро) → --stage 2 (данные) → обе модели.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport os\nimport sys\n\nimport numpy as np\nimport torch\n\n_HERE = os.path.dirname(os.path.abspath(__file__))\nif _HERE not in sys.path:\n    sys.path.insert(0, _HERE)\nimport exp47_common as C  # noqa: E402\n\nEPS_LFM, EPS_QWEN = 1e-5, 1e-6\nRNG = np.random.default_rng(47)\nLAGS = (1, 2, 4, 8, 16, 32)\n\n\ndef med(x):\n    x = np.asarray(x, float)\n    x = x[np.isfinite(x)]\n    return round(float(np.median(x)), 4) if len(x) else None\n\n\ndef boot_ci(v, B=2000):\n    v = np.asarray(v, float); v = v[np.isfinite(v)]\n    if len(v) < 3:\n        return [None, None]\n    # seed фиксирован НА ВЫЗОВ (ревью §2.6): CI не зависит от порядка вызовов\n    idx = np.random.default_rng(47).integers(0, len(v), (B, len(v)))\n    m = np.median(v[idx], axis=1)\n    return [round(float(np.quantile(m, 0.025)), 4), round(float(np.quantile(m, 0.975)), 4)]\n\n\n# ------------------------------------------------------------------ каркас метрик\n\ndef frame_in_metric(J, r, L):\n    """Базис top-r инвариантного подпространства JᵀJ, G-ортонормированный.\n    Фикс (ревью E49 §1.1): torch.linalg.cholesky даёт G = L·Lᵀ (НЕ LᵀL), значит\n    ‖x‖_G = ‖Lᵀx‖₂ и G-ортонорм — qr(LᵀV), а не qr(LV). Возвращаемый базис —\n    Lᵀ·span(V) (НЕ span(V)); углы между подпространствами в G-метрике равны\n    евклидовым углам между Lᵀ-образами — это то, зачем встроивание нужно.\n    ПРЕЖНИЕ M-ячейки сетки\n    exp49_leaf_info_*.json считали метрику LᵀL ≠ G — НЕИНТЕРПРЕТИРУЕМЫ (евклидовы\n    H2-выводы exp49b2 не затронуты: там L=None)."""\n    V = C.V16_from_J(J, r=r)[0]                       # [D,r] ортонорм (евклид)\n    if L is None:\n        return V\n    Wm = L.T @ V\n    return torch.linalg.qr(Wm)[0]\n\n\ndef tau_triplets(Vs):\n    a = [C.frame_kinematics(Vs[t], Vs[t + 1])[0] for t in range(len(Vs) - 2)]\n    b = [C.frame_kinematics(Vs[t + 1], Vs[t + 2])[0] for t in range(len(Vs) - 2)]\n    c = [C.frame_kinematics(Vs[t], Vs[t + 2])[0] for t in range(len(Vs) - 2)]\n    return [(x + y - z) / x for x, y, z in zip(a, b, c) if x > 1e-6]\n\n\ndef v1_signed_series(us, A, eps):\n    out, prev = [], None\n    for u in us:\n        J, _, _ = C.J_analytic(A, u, eps)\n        G = J @ J.T; G = 0.5 * (G + G.T)\n        lam, Ue = torch.linalg.eigh(G)\n        v1 = J.T @ Ue[:, -1]\n        v1 = v1 / v1.norm()\n        if prev is not None and float(v1 @ prev) < 0:\n            v1 = -v1\n        out.append(v1); prev = v1\n    return out\n\n\ndef v1_lag(v1s):\n    return {L: [float(torch.acos((v1s[t] @ v1s[t + L]).abs().clamp(max=1 - 1e-12)))\n                for t in range(len(v1s) - L)] for L in LAGS}\n\n\ndef holonomy(Vs, L=8):\n    out = []\n    r = Vs[0].shape[1]\n    for t in range(len(Vs) - L):\n        P = torch.eye(r, dtype=torch.float64)\n        for k in range(L):\n            P = P @ C.procrustes_R(Vs[t + k], Vs[t + k + 1])\n        P = C.procrustes_R(Vs[t + L], Vs[t]) @ P\n        th, _, _ = C.schur_angles(P)\n        out.append(float(torch.sqrt((th * th).sum())))\n    return out\n\n\ndef doc_stats(Vs, v1s=None):\n    d = {"s": {str(L): [C.frame_kinematics(Vs[t], Vs[t + L])[0] for t in range(len(Vs) - L)]\n               for L in LAGS},\n         "tau": tau_triplets(Vs)}\n    if v1s is not None:\n        d["v1"] = {str(L): v for L, v in v1_lag(v1s).items()}\n    return d\n\n\ndef agg(docss):\n    out = {"s_lag": {str(L): med([x for d in docss for x in d["s"][str(L)]]) for L in LAGS},\n           "tau": med([x for d in docss for x in d["tau"]])}\n    if docss and "v1" in docss[0]:\n        out["v1_lag"] = {str(L): med([x for d in docss for x in d["v1"][str(L)]]) for L in LAGS}\n    # bootstrap по документам на τ и на s(1)\n    taus = [med(d["tau"]) for d in docss]\n    out["tau_doc_boot95"] = boot_ci(taus)\n    return out\n\n\n# ------------------------------------------------------------------ контроли\n\ndef nz(x):\n    return x / x.norm()\n\n\ndef control_white(T, D):\n    g = torch.Generator().manual_seed(7)\n    return [nz(torch.randn(D, dtype=torch.float64, generator=g))\n            for _ in range(T)]\n\n\ndef control_coherent(T, D, delta=0.02):\n    """ГЕОДЕЗИЧЕСКИЙ контроль (фикс ревью E49-4 §2.1 — та же ортогонализация, что в\n    synth_coh exp49b2; прежняя версия без w⟂u давала невалидный CB)."""\n    g = torch.Generator().manual_seed(11)\n    u = nz(torch.randn(D, dtype=torch.float64, generator=g))\n    w = nz(torch.randn(D, dtype=torch.float64, generator=g))\n    w = nz(w - u * (u @ w))\n    out = []\n    for t in range(T):\n        out.append(u.clone())\n        u2 = math.cos(delta) * u + math.sin(delta) * w\n        u = nz(u2)\n        z = nz(torch.randn(D, dtype=torch.float64, generator=g))\n        w = nz(w + 0.05 * (z - (z @ u) * u - (z @ w) * w))\n        w = nz(w - u * (u @ w))\n    return out\n\n\ndef run_stage1(A, eps, D, T=96):\n    res = {}\n    for name, us in [("C-A_white", control_white(T, D)), ("C-B_coherent", control_coherent(T, D))]:\n        Vs = [frame_in_metric(C.J_analytic(A, u, eps)[0], 16, None) for u in us]\n        v1s = v1_signed_series(us, A, eps)\n        d = doc_stats(Vs, v1s)\n        res[name] = {"s_lag": {k: med(v) for k, v in d["s"].items()},\n                     "tau": med(d["tau"]),\n                     "v1_lag": {k: med(v) for k, v in d["v1"].items()}}\n    ok = (res["C-A_white"]["tau"] > 0.7 and res["C-B_coherent"]["tau"] < 0.5\n          and res["C-B_coherent"]["s_lag"]["16"] and res["C-B_coherent"]["s_lag"]["1"]\n          and res["C-B_coherent"]["s_lag"]["16"] > res["C-B_coherent"]["s_lag"]["1"])\n    res["SELFTEST"] = "ok" if ok else "ВАЛИТСЯ: конвейер не различает белое/связное"\n    return res\n\n\n# ------------------------------------------------------------------ данные\n\ndef smoothed_us(u, nrm, W):\n    """Сглаживание шагов (причинное окно). ИЗВЕСТНОЕ ОГРАНИЧЕНИЕ (ревью E49 §2.7):\n    на первых W−1 позициях окно короче — краевой эффект ~(W−1)/T (~4% при W=16 на\n    T=384); в исторических прогонах оставлено как есть; новые скрипты (e52) должны\n    отбрасывать первые W−1 или брать центрированное окно."""\n    h = u * torch.tensor(nrm, dtype=torch.float64).unsqueeze(1)\n    if W > 1:\n        hs = torch.stack([h[max(0, t - W + 1):t + 1].mean(0) for t in range(len(h))])\n    else:\n        hs = h\n    return [hs[t] / hs[t].norm() for t in range(len(hs))]\n\n\ndef load_model(args):\n    from transformers import AutoConfig, AutoModelForCausalLM  # type: ignore\n    if args.model == "qwen":\n        mp = os.environ.get("PRISM_QWEN_DIR", "/data/Qwen/Qwen3.5-0.8B")\n        ck = torch.load(os.environ.get("PRISM_JCACHE",\n                        "/data/prism_kernels/exp27b/results/exp27b_jcache_L23.pt"),\n                        map_location="cpu", weights_only=False)\n        U = ck["U128"]\n        U = U.tolist() if torch.is_tensor(U) else list(U)\n        cfg = AutoConfig.from_pretrained(mp)\n        m = AutoModelForCausalLM.from_pretrained(mp, dtype=torch.float32).eval()\n        g = 1.0 + m.model.norm.weight.detach().float()\n        npz, jp, eps = (os.environ.get("PRISM_NPZ",\n                        "/data/prism_kernels/exp47b/exp47b_gen_kinematics.npz"),\n                        os.environ.get("PRISM_DOCS_JSON",\n                                       "results/exp47b_gen_kinematics.json"), EPS_QWEN)\n    else:\n        from transformers.cache_utils import LinearAttentionLayer  # type: ignore\n        mp = os.environ.get("PRISM_LFM3_DIR", "/data/LiquidAI/LFM3-1.7B")\n        cfg = AutoConfig.from_pretrained(mp, trust_remote_code=True)\n        cfg.layer_types = list(json.load(open(os.path.join(mp, "config.json")))["layer_types"])\n        _o = LinearAttentionLayer.update_conv_state\n        LinearAttentionLayer.update_conv_state = (\n            lambda self, cs, si=0, **kw: _o(self, cs.squeeze(1) if cs.ndim == 4 else cs, si, **kw))\n        m = AutoModelForCausalLM.from_pretrained(mp, config=cfg, dtype=torch.float32,\n                                                 trust_remote_code=True).eval()\n        g = m.model.embedding_norm.weight.detach().float()\n        U = list(range(0, 384, 3))[:128]\n        npz, jp, eps = ("/data/prism_kernels/exp48b/exp48b_lfm3_gen_kinematics.npz",\n                        "results/exp48b_lfm3_gen_kinematics.json", EPS_LFM)\n    W_head = m.get_input_embeddings().weight.detach()\n    # tied-проверка универсальная (ревью E49 §1.4/1.6): у LFM3-1.7B\n    # tie_word_embeddings=True и lm_head в чекпоинте ОТСУТСТВУЕТ (проверено\n    # 2026-11-06 по config/safetensors) — head ≡ embeddings, A был верным;\n    # для untied-моделей берём output head.\n    head = getattr(m, "lm_head", None)\n    if head is not None and head.weight.data_ptr() != W_head.data_ptr():\n        W_head = head.weight.detach()\n    A = C.head_M(W_head.float()[U], g)\n    load_model.last_g = g          # для e52a: пересборка A из изменённых строк head\n    return A, U, eps, npz, jp\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--stage", type=int, default=2)\n    ap.add_argument("--model", choices=("qwen", "lfm3"), default="qwen")\n    args = ap.parse_args()\n    root = os.path.join(_HERE, "..")\n    A, U, eps, npz, jp = load_model(args)\n    D = int(A.shape[1])\n    out = {"model": args.model, "U_head": U[:6], "eps": eps}\n\n    if args.stage == 1:\n        out["controls"] = run_stage1(A, eps, D)\n        print("CONTROLS:", json.dumps(out["controls"], indent=1), flush=True)\n        if out["controls"]["SELFTEST"] != "ok":\n            sys.exit("selftest failed")\n\n    if args.stage >= 2:\n        G = A.T @ A\n        Lch = torch.linalg.cholesky(G + 1e-10 * torch.eye(D, dtype=torch.float64) * float(G.diag().mean()))\n        docs = json.load(open(os.path.join(root, jp)))["docs"]\n        z = np.load(npz, allow_pickle=True)\n        ud = {}\n        for doc in docs:\n            f = z[f"{doc[\'dom\']}_{doc[\'si\']}"].item()\n            ud[f"{doc[\'dom\']}_{doc[\'si\']}"] = (\n                torch.from_numpy(z[f"{doc[\'dom\']}_{doc[\'si\']}_u"].astype(np.float64)),\n                np.asarray(f["hnorm"], dtype=float))\n        rep = {}\n        # стриминг: один (W, doc) в памяти; для него eig один раз, все (r,metric) из него\n        comb = {}\n        for W in (1, 4, 16):\n            for key, (u, nrm) in ud.items():\n                us = smoothed_us(u, nrm, W)\n                eig = []\n                for x in us:\n                    J, _, _ = C.J_analytic(A, x, eps)\n                    Gm = J @ J.T; Gm = 0.5 * (Gm + Gm.T)\n                    lam, Ue = torch.linalg.eigh(Gm)\n                    eig.append((J, Ue))\n                for r in (1, 2, 4, 16):\n                    for metric in ("eucl", "M"):\n                        Vs = []\n                        for (J, Ue) in eig:\n                            V = J.T @ Ue[:, -r:]\n                            V = V / V.norm(dim=0, keepdim=True)\n                            V = torch.linalg.qr(V)[0]\n                            if metric == "M":\n                                # ревью E49-2 §2.1: cholesky даёт G=L·Lᵀ ⇒ G-ортонорм\n                                # = qr(LᵀV); инлайн-ветка ранее крутила метрику LᵀL\n                                V = torch.linalg.qr(Lch.T @ V)[0]\n                            Vs.append(V)\n                        comb.setdefault(f"W{W}_r{r}_{metric}", []).append(doc_stats(Vs, None))\n                del eig\n            print(f"W{W} done", flush=True)\n        for k, ds in comb.items():\n            rep[k] = agg(ds)\n            print(f"{k}: tau={rep[k][\'tau\']} s1={rep[k][\'s_lag\'][\'1\']} "\n                  f"s32={rep[k][\'s_lag\'][\'32\']}", flush=True)\n        # v1-метрики отдельно (W=1): угол между v1 со знакомым транспортом\n        v1ds = {}\n        for key, (u, nrm) in ud.items():\n            us = smoothed_us(u, nrm, 1)\n            v1ds[key] = v1_signed_series(us, A, eps)\n        rep["v1"] = {str(L): med([float(torch.acos((v1ds[k][t] @ v1ds[k][t + L]).abs().clamp(max=1 - 1e-12)))\n                                 for k in v1ds for t in range(len(v1ds[k]) - L)]) for L in LAGS}\n        out["grid"] = rep\n\n    path = os.path.join(root, "results", f"exp49_leaf_info_{args.model}.json")\n    os.makedirs(os.path.dirname(path), exist_ok=True)\n    json.dump(out, open(path, "w"), indent=1)\n    print("WROTE", path)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'experiments/exp52a_perturb_axis.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('#!/usr/bin/env python\n"""E-52a PREFLIGHT: ось отклонения τ = τ* + ε (E52_review, тесты A–C).\n\nВопрос не «болеет ли», а «куда и на сколько отклоняется τ под батареей\nвозмущений». Нули/метрика — E-49b (W4, paired по документам, τ-медиана).\n\nВозмущения (дозные семейства, всё детерминировано по сидам):\n  FLAT  : готовые строки головы E-45 (flat_l0.01 < flat_l0.1 < flat_noleash,\n          доза = глубина уплощения; qwen-only — кэш exp45_flat_rows.pt);\n  RTN   : симметричная per-row квантизация A, int8 → int4 → int2 (доза = биты);\n  NOISE : мультипликативный шум строк A, σ = 0.005/0.02/0.08 (реад-аут);\n  U-NS  : шум скрытых u перед сглаживанием, σ = 0.01/0.02/0.04 (трэнк-прокси).\nТесты: A — знак Δτ (одно семейство ⇒ знак однообразен?); B — монотонность\nпо дозе (Spearman); C — перенос на вторую модель (сравнение потом по JSON).\n\nК администратору чисел: τ считается top-16 РЕБРАМИ J через спектр JJᵀ [128,128]\n(ранг ≤128 ⇒ тождественно [D,D]-варианту exp49b2, в ~50 раз быстрее); T=96,\nW=4 — самосогласовано внутри прогона, с baseline E-49b (T=384) абсолютные\nуровни не сравниваем — только paired-Δ.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nimport sys\n\nimport numpy as np\nimport torch\n\n_HERE = os.path.dirname(os.path.abspath(__file__))\nif _HERE not in sys.path:\n    sys.path.insert(0, _HERE)\nimport exp47_common as C  # noqa: E402\nfrom exp49_leaf_info import load_model, smoothed_us, tau_triplets, boot_ci  # noqa: E402\n\nTMAX = 96\nW = 4\n\n\ndef _seed(s):\n    return int.from_bytes(hashlib.sha256(s.encode()).digest()[:8], "little") % (1 << 62)\n\n\ndef rtn(A, bits):\n    """Симметричная per-row RTN: scale = row-absmax/(2^(b-1)-1)."""\n    lev = 2 ** (bits - 1) - 1\n    sc = A.abs().amax(dim=1, keepdim=True).clamp(min=1e-12) / lev\n    return (torch.clamp((A / sc).round(), -lev, lev) * sc)\n\n\ndef frames16(A, us, eps):\n    """Top-16 рам листа через спектр JJᵀ [K,K] (ранг J ≤ K=128 < D)."""\n    out = []\n    for x in us:\n        J, _, _ = C.J_analytic(A, x, eps)\n        Gs = J @ J.T\n        Gs = 0.5 * (Gs + Gs.T)\n        _, Ue = torch.linalg.eigh(Gs)\n        V = J.T @ Ue[:, -16:]\n        out.append(torch.linalg.qr(V / V.norm(dim=0, keepdim=True))[0])\n    return out\n\n\ndef tau_of(A, us, eps):\n    t = tau_triplets(frames16(A, us, eps))\n    return float(np.median(t)) if t else None\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--model", choices=("qwen", "lfm3"), default="qwen")\n    ap.add_argument("--n-docs", type=int, default=32)\n    ap.add_argument("--families", default="flat,rtn,noise,u_ns")\n    ap.add_argument("--npz", default=None); ap.add_argument("--docs-json", default=None)\n    ap.add_argument("--model-dir", default=None,\n                    help="пересобрать A из этого чекпойнта (тест D, без full-load: safetensors slice)")\n    ap.add_argument("--out", default=None)\n    args = ap.parse_args()\n    root = os.path.join(_HERE, "..")\n    fams = set(args.families.split(","))\n\n    ns = argparse.Namespace(model=args.model)\n    A0, U, eps, npz, jp = load_model(ns)\n    g = load_model.last_g\n    if args.npz:\n        npz = args.npz\n    docs = json.load(open(args.docs_json or os.path.join(root, jp)))["docs"][: args.n_docs]\n    z = np.load(npz, allow_pickle=True)\n    if args.model_dir:      # тест D: A/g/eps от модели самой трассы\n        import glob\n        from safetensors import safe_open\n        mp = args.model_dir\n        files = sorted(glob.glob(os.path.join(mp, "*.safetensors")))\n        HCAND = ("lm_head.weight", "model.language_model.lm_head.weight",\n                 "model.embed_tokens.weight", "model.language_model.embed_tokens.weight")\n        NCAND = ("model.norm.weight", "model.language_model.norm.weight")\n        head = normw = None\n        for f in files:\n            with safe_open(f, framework="pt") as sf:\n                ks = set(sf.keys())\n                if head is None:\n                    head = next((sf.get_tensor(c) for c in HCAND if c in ks), None)\n                if normw is None:\n                    normw = next((sf.get_tensor(c) for c in NCAND if c in ks), None)\n            if head is not None and normw is not None:\n                break\n        assert head is not None and normw is not None, f"head/norm не найдены в {mp}"\n        wn = normw.float().double()\n        if "g_norm" in z.files:          # свежая трасса: единый источник истины\n            g = torch.from_numpy(z["g_norm"].astype(np.float64))\n            eps = float(z["eps"])\n            print("[model-dir] g/eps из npz", flush=True)\n        else:                            # старая трасса: статистика w (w~0 ⇒ 1+w; w~1 ⇒ w)\n            g = wn if float(wn.abs().mean()) > 0.5 else 1.0 + wn\n            cfg = json.load(open(os.path.join(mp, "config.json")))\n            eps = float(cfg.get("rms_norm_eps")\n                        or cfg.get("text_config", {}).get("rms_norm_eps") or 1e-6)\n            mode = "w" if float(wn.abs().mean()) > 0.5 else "1+w"\n            print(f"[model-dir] fallback: g={mode}, eps={eps}, |w|mean={float(wn.abs().mean()):.3f}", flush=True)\n        A0 = C.head_M(head.float()[U], g)\n\n    base_us = {}\n    for d in docs:\n        k = f"{d[\'dom\']}_{d[\'si\']}"\n        u = torch.from_numpy(z[k + "_u"].astype(np.float64))[:TMAX]\n        hn = np.asarray(z[k].item()["hnorm"], dtype=float)[:TMAX]\n        base_us[k] = (u, hn)\n        # sanity: рамочный маршрут 128-трюка против [D,D] — один раз\n    if True:\n        k0 = next(iter(base_us)); u0, hn0 = base_us[k0]\n        us0 = smoothed_us(u0, hn0, W)\n        J, _, _ = C.J_analytic(A0, us0[5], eps)\n        V_small = J.T @ torch.linalg.eigh(J @ J.T)[1][:, -16:]\n        V_big = (J.T @ J) @ torch.linalg.eigh((J.T @ J).double())[1][:, -16:]  # [D,D]\n        ang = torch.acos(torch.linalg.svdvals(\n            torch.linalg.qr(V_small)[0].T @ torch.linalg.qr(V_big)[0]).clamp(-1 + 1e-12, 1 - 1e-12))\n        assert float(ang.max()) < 1e-4, f"128-трюк расходится с [D,D]: {float(ang.max())}"  # 1.4e-6实测\n\n    variants = [("base", A0, None)]\n    if "flat" in fams and args.model == "qwen":\n        fl = torch.load(os.path.join(root, "results", "exp45_flat_rows.pt"),\n                        map_location="cpu", weights_only=False)\n        for nm in ("flat_l0.01", "flat_l0.1", "flat_noleash"):\n            variants.append((nm, C.head_M(fl[nm].float(), g), None))\n    if "rtn" in fams:\n        for b in (8, 4, 2):\n            variants.append((f"rtn{b}", rtn(A0, b), None))\n    if "noise" in fams:\n        for s in (0.005, 0.02, 0.08):\n            xi = torch.randn(A0.shape, dtype=torch.float64,\n                             generator=torch.Generator().manual_seed(_seed(f"An{s}")))\n            variants.append((f"noise{s}", A0 * (1.0 + s * xi), None))\n    u_variants = ([("u_ns0.01", 0.01), ("u_ns0.02", 0.02), ("u_ns0.04", 0.04)]\n                  if "u_ns" in fams else [])\n\n    out = {"model": args.model, "W": W, "T": TMAX, "n_docs": len(docs),\n           "note": "paired Δτ (variant−base) per doc; 128-трюк верифицирован против [D,D]"}\n    taus = {}\n    for nm, A, _ in variants:\n        vals = {}\n        for k, (u, hn) in base_us.items():\n            vals[k] = tau_of(A, smoothed_us(u, hn, W), eps)\n        taus[nm] = vals\n        print(nm, "done", flush=True)\n    for nm, s in u_variants:\n        xi = torch.randn((1, A0.shape[1]), dtype=torch.float64,\n                         generator=torch.Generator().manual_seed(_seed(f"u{nm}")))\n        vals = {}\n        for k, (u, hn) in base_us.items():\n            un = u + s * u.norm(dim=1, keepdim=True) * xi.to(u.dtype)\n            vals[k] = tau_of(A0, smoothed_us(un, hn, W), eps)\n        taus[nm] = vals\n        print(nm, "done", flush=True)\n\n    dom = {k: k.split("_")[0] for k in base_us}\n    summ = {}\n    for nm in taus:\n        if nm == "base":\n            summ[nm] = {"tau": float(np.median(list(taus["base"].values())))}\n            continue\n        dd = [taus[nm][k] - taus["base"][k] for k in base_us\n              if taus[nm][k] is not None and taus["base"][k] is not None]\n        ddom = {}\n        for d0 in set(dom.values()):\n            sub = [taus[nm][k] - taus["base"][k] for k in base_us\n                   if dom[k] == d0 and taus[nm][k] is not None]\n            ddom[d0] = float(np.median(sub)) if sub else None\n        summ[nm] = {"d_tau": float(np.median(dd)), "ci": boot_ci(dd),\n                    "dom_sign_pos": sum(1 for v in ddom.values() if (v or 0) > 0),\n                    "n_dom": len(ddom), "per_dom": ddom,\n                    "tau": float(np.median(list(taus[nm].values())))}\n        print(nm, summ[nm]["d_tau"], summ[nm]["ci"], flush=True)\n    out["per_doc"] = taus\n    out["summary"] = summ\n    path = args.out or os.path.join(root, "results", f"exp52a_perturb_{args.model}.json")\n    json.dump(out, open(path, "w"), indent=1, allow_nan=False)\n    print("WROTE", path)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/__init__.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""PRISM: Jacobian-based analysis of transformer models.\n\nPublic API. Import from ``prism`` rather than submodules when possible.\nBackward-compatible: ``from prism.jacobian import ...`` continues to work.\n"""\n\nfrom .bridge import jacobian_gap\nfrom .esg import (\n    compute_esg,\n    compute_esg_per_sample,\n    cross_positional_gramian,\n)\nfrom .jacobian import (\n    compute_jacobian_for_tokens,\n    compute_hidden_jacobian,\n    compute_hidden_jacobian_rows,\n    softmax_jacobian,\n    uniform_centered,\n    bridge_rows_backward,\n    h_norm_at_perturbed_h,\n)\nfrom .calibration import (\n    load_calibration_data,\n    extract_embeddings,\n    uniform_layer_mapping,\n    get_layer_types,\n    build_global_token_set,\n)\n# NOTE (audit docs/changed_review.md, 2026-07-18): prism.calibration imports\n# `datasets` at module level, so `import prism` REQUIRES `datasets` to be\n# installed (pre-v6.0 __init__ was docstring-only and had no such dependency).\n# Deliberately NOT wrapped in try/except and NOT made a lazy import:\n# `load_calibration_data` is a core entry point of the public API, and a\n# silent absence would surface as a confusing NameError deep in a\n# long-running experiment. If `datasets` is missing, `import prism` fails\n# with a clear ModuleNotFoundError at install/import time. To use the rest\n# of the package without it: `pip install datasets` (it is small).\n\nfrom .diagnostics import (\n    run_layer_diagnostics,\n    run_identity_test,\n    run_heatmap,\n)\nfrom .layer_selection import select_pairs, classify_pair\nfrom .cache_io import find_targets_dir\nfrom .cache import (\n    SVDCache,\n    MetricCache,\n    bootstrap_ci,\n    bootstrap_ci_paired,\n    deterministic_mode,\n    iter_position_pairs,\n)\nfrom .spectral import (\n    top_r_svd,\n    spectrum_from_gramian,\n    eigengap,\n    eigengap_min,\n    entropy_effective_rank,\n    stable_rank,\n    participation_ratio,\n    snr_top1_vs_rest,\n    snr_e14,\n    snr_e14_parts,\n    gauge_share,\n    gauge_share_top1,\n)\nfrom .holonomy import (\n    procrustes_align,\n    holonomy_accumulate,\n    holonomy_spectrum,\n    geodesic_curvature,\n    eigenframe_basis,\n    holonomy_along_path,\n    holonomy_along_2d_loop,\n)\nfrom .jstruct import (\n    build_M_K,\n    decompose_w_vs_uhat,\n    v_pred_from_JT,\n)\nfrom .interventions import (\n    EditSpec,\n    make_base,\n    make_readout_pull,\n    make_random_ortho,\n    make_random_matched,\n    make_tail_only,\n    apply_edit,\n    apply_edit_and_cache,\n)\nfrom .noise import (\n    calibrate_noise_floor,\n    validate_signal,\n)\nfrom .permutation import (\n    permutation_max_statistic,\n    generate_null,\n    cluster_based_permutation,\n)\nfrom .topology import (\n    zigzag_persistence,\n    project_to_subspace,\n    project_to_pca_subspace,\n)\nfrom .pipeline import (\n    ExperimentSpec,\n    OutputArtifacts,\n    run_spec,\n    run_many,\n    validate_spec,\n    load_artifacts,\n    default_output_subdir,\n)\n\n__all__ = [\n    "jacobian_gap",\n    "compute_esg",\n    "compute_esg_per_sample",\n    "cross_positional_gramian",\n    "compute_jacobian_for_tokens",\n    "compute_hidden_jacobian",\n    "compute_hidden_jacobian_rows",\n    "softmax_jacobian",\n    "uniform_centered",\n    "bridge_rows_backward",\n    "h_norm_at_perturbed_h",\n    "load_calibration_data",\n    "extract_embeddings",\n    "uniform_layer_mapping",\n    "get_layer_types",\n    "build_global_token_set",\n    "run_layer_diagnostics",\n    "run_identity_test",\n    "run_heatmap",\n    "select_pairs",\n    "classify_pair",\n    "find_targets_dir",\n    "SVDCache",\n    "MetricCache",\n    "bootstrap_ci",\n    "bootstrap_ci_paired",\n    "deterministic_mode",\n    "iter_position_pairs",\n    "top_r_svd",\n    "spectrum_from_gramian",\n    "eigengap",\n    "eigengap_min",\n    "entropy_effective_rank",\n    "stable_rank",\n    "participation_ratio",\n    "snr_top1_vs_rest",\n    "snr_e14",\n    "snr_e14_parts",\n    "gauge_share",\n    "gauge_share_top1",\n    "procrustes_align",\n    "holonomy_accumulate",\n    "holonomy_spectrum",\n    "geodesic_curvature",\n    "eigenframe_basis",\n    "holonomy_along_path",\n    "holonomy_along_2d_loop",\n    "build_M_K",\n    "decompose_w_vs_uhat",\n    "v_pred_from_JT",\n    "EditSpec",\n    "make_base",\n    "make_readout_pull",\n    "make_random_ortho",\n    "make_random_matched",\n    "make_tail_only",\n    "apply_edit",\n    "apply_edit_and_cache",\n    "calibrate_noise_floor",\n    "validate_signal",\n    "permutation_max_statistic",\n    "generate_null",\n    "cluster_based_permutation",\n    "zigzag_persistence",\n    "project_to_subspace",\n    "project_to_pca_subspace",\n    "ExperimentSpec",\n    "OutputArtifacts",\n    "run_spec",\n    "run_many",\n    "validate_spec",\n    "load_artifacts",\n    "default_output_subdir",\n]\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/calibration.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Calibration: data loading, tokenization, global token set, layer mapping."""\n\nimport hashlib\nimport json\nimport os\nimport re\nimport warnings\nfrom collections import Counter\nfrom pathlib import Path\n\nimport torch\nfrom datasets import load_dataset\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\n# --- закрепление пула (PLAN «Запрос 11»: все результаты воспроизводимы) -------------------\n# Порядок streaming-потока на Hub ничем не закреплён, поэтому «первые подходящие строки»\n# меняются от прогона к прогону: E-0.1 при идентичных params дал другой top-token набор\n# (pk_hash 5c12c174… -> 30d82ade…) и сдвинул C_b(sigma_d) до x31. Пин = сырые тексты принятых\n# строк + отпечаток; при наличии пина поток не читается вообще, значит прогон воспроизводим\n# и офлайн. Тексты (не тензоры): input_ids по-прежнему производит токенизатор.\nPIN_DIR_ENV = "PRISM_CALIB_PIN_DIR"\nPIN_WRITE_ENV = "PRISM_CALIB_PIN_WRITE"      # =1: разрешить записать новый пин\nPIN_REQUIRE_ENV = "PRISM_CALIB_REQUIRE_PIN"  # =1: отсутствие пина = громкий отказ\nLAST_POOL_FPS: dict[str, str] = {}           # domain -> отпечаток последнего загруженного пула\n\n\ndef pool_fingerprint(texts: list[str]) -> str:\n    """Отпечаток пула по сырым текстам (от токенизатора не зависит)."""\n    h = hashlib.sha256()\n    for t in texts:\n        h.update(t.encode("utf-8"))\n        h.update(b"\\x00")\n    return h.hexdigest()[:16]\n\n\ndef _default_pin_dir() -> Path | None:\n    """Каталог пинов по умолчанию, или None если искать негде.\n\n    В flattening-Kaggle-ядре модулей нет на диске и `__file__` не определён (упавший exp3:\n    `NameError: name \'__file__\' is not defined` прямо в `_pin_path`) — значит путь репозитория\n    строится только когда модуль реально лежит в репозитории; иначе пин ищется по `PIN_DIR_ENV`,\n    а без него — поток (или громкий отказ при `PIN_REQUIRE_ENV`).\n    """\n    try:\n        here = Path(__file__).resolve()\n    except NameError:\n        return None\n    return here.parent.parent / "data" / "calib_pins"\n\n\ndef _pin_path(domain, model_name, n_samples, min_length, max_length,\n              save_offset_mapping) -> Path | None:\n    root = os.environ.get(PIN_DIR_ENV)\n    base = Path(root) if root else _default_pin_dir()\n    if base is None:\n        return None\n    slug = re.sub(r"[^0-9A-Za-z.]+", "_", model_name)\n    mode = "offset" if save_offset_mapping else "plain"\n    return base / f"{domain}__{slug}__n{n_samples}__min{min_length}__max{max_length}__{mode}.json"\n\n\ndef uniform_layer_mapping(n_layers_large: int, n_layers_small: int) -> dict[int, int]:\n    """Uniform layer mapping from large to small model layers."""\n    mapping = {}\n    for i in range(n_layers_large):\n        j = round(i * (n_layers_small - 1) / (n_layers_large - 1))\n        mapping[i] = j\n    return mapping\n\n\ndef get_layer_types(model: AutoModelForCausalLM) -> list[str]:\n    """Get layer types from model config (full_attention / linear_attention)."""\n    cfg = model.config\n    return list(\n        cfg.layer_types if hasattr(cfg, "layer_types") else cfg.text_config.layer_types\n    )\n\n\n_DOMAIN_DATASETS = {\n    "wiki": ("allenai/c4", "en", "train", "text"),\n    "math": ("nvidia/OpenMathInstruct-2", None, "train", "problem"),\n    "code": ("neyralabs/apex-agents", None, "train", "prompt"),\n    "chat": ("HuggingFaceH4/ultrachat_200k", None, "train_sft", "messages"),\n}\n\n\ndef _load_dataset_for_domain(\n    domain: str,\n) -> tuple[str, str | None, str, str]:\n    """Return (dataset_id, config, split, text_field) for a domain.\n\n    Falls back to wiki (C4) if the target dataset is unavailable, with a\n    RuntimeWarning: a fallback run silently poisons cross-domain comparisons\n    (E-13/E-27 T_domain), so the substitution must be visible in the log.\n    """\n    ds_id, cfg, split, field = _DOMAIN_DATASETS[domain]\n    try:\n        kw: dict[str, object] = {"split": split, "streaming": True}\n        if cfg is not None:\n            kw["name"] = cfg\n        ds = load_dataset(ds_id, **kw)  # type: ignore[call-overload]\n        next(iter(ds))  # force the first fetch: streaming is lazy, a broken/\n        # gated dataset would otherwise pass the check and fail inside\n        # load_calibration_data (loud crash instead of the intended fallback).\n        return ds_id, cfg, split, field\n    except Exception:\n        warnings.warn(\n            f"domain {domain} ({ds_id}) unavailable — falling back to wiki/C4. "\n            f"Domain labels in this run are INVALID for cross-domain "\n            f"comparisons (E-13/E-27).",\n            RuntimeWarning,\n            stacklevel=2,\n        )\n        return _DOMAIN_DATASETS["wiki"]\n\n\ndef _extract_text_from_sample(\n    sample: dict,\n    field: str,\n) -> str:\n    """Extract text from a dataset sample, handling different formats."""\n    if field == "messages":\n        # Chat format: list of {role, content}\n        texts = [m.get("content", "") for m in sample.get("messages", [])]\n        return " ".join(texts)\n    val = sample.get(field, "")\n    if isinstance(val, list):\n        # e.g. code_contests solutions: list of strings\n        return "\\n".join(str(v) for v in val)\n    if isinstance(val, dict):\n        return str(val)\n    return val\n\n\ndef load_calibration_data(\n    model_name: str,\n    max_length: int = 64,\n    min_length: int = 64,\n    n_samples: int = 4,\n    domain: str = "wiki",\n    save_offset_mapping: bool = False,\n) -> list[dict[str, torch.Tensor]]:\n    """Load and tokenize calibration data.\n\n    Defaults match the tested protocol (paper §12/C.6: S=64, filter >= 64,\n    N=4); experiments pass explicit lengths. domain: \'wiki\' (C4), \'math\'\n    (OpenMathInstruct-2), \'code\' (apex-agents), \'chat\' (ultrachat). Falls back\n    to wiki if the target dataset is unavailable. save_offset_mapping: adds\n    `offset_mapping` to each sample (E-31 sentence boundaries).\n    """\n    tokenizer = AutoTokenizer.from_pretrained(model_name)\n    pin = _pin_path(domain, model_name, n_samples, min_length, max_length, save_offset_mapping)\n    texts: list[str] | None = None\n    ds_id = cfg = split = field = None\n    if pin is not None and pin.exists():\n        meta = json.loads(pin.read_text(encoding="utf-8"))\n        texts = list(meta["texts"])\n        print(f"[calib] {domain}: пин {pin.name}, строк={len(texts)}, "\n              f"pool_fp={meta[\'fp\']}, поток не читается", flush=True)\n    elif os.environ.get(PIN_REQUIRE_ENV) == "1":\n        raise RuntimeError(\n            f"нет пина пула {pin or f\'каталог пинов недоступен — задайте {PIN_DIR_ENV}\'} при "\n            f"{PIN_REQUIRE_ENV}=1: прогон на потоке с Hub недопустим, "\n            f"иначе числа не воспроизводимы. Сгенерируйте пин с {PIN_WRITE_ENV}=1 один раз и "\n            f"закоммитьте его."\n        )\n    else:\n        ds_id, cfg, split, field = _load_dataset_for_domain(domain)\n        kw: dict[str, object] = {"split": split, "streaming": True}\n        if cfg is not None:\n            kw["name"] = cfg\n        dataset = load_dataset(ds_id, **kw)  # type: ignore[call-overload]\n    samples = []\n    # NOTE (audit4 Q9): the offset-mapping path tokenizes with\n    # add_special_tokens=False, the plain path with the tokenizer default (True).\n    # Deliberate, not a bug: offset_mapping must stay aligned 1:1 with the text\n    # span (special tokens carry no span), and the two paths feed different\n    # consumers. Consequence to keep in mind: a token set / calibration built\n    # through the offset path is NOT the same sequence as the mainline E-16\n    # calibration of the same text — do not mix their token sets.\n    accepted: list[str] = []\n    # Из пина идут уже извлечённые тексты (извлечение = _extract_text_from_sample на момент\n    # записи пина); с потока идут сырые строки.\n    rows = ((t, None) for t in texts) if texts is not None else ((None, raw) for raw in dataset)\n    for pinned_text, raw in rows:\n        text = pinned_text if pinned_text is not None else _extract_text_from_sample(raw, field)\n        if not text:\n            continue\n        if save_offset_mapping:\n            # Токенизатор вызывается на ОДНОЙ строке, поэтому input_ids — плоский список id, а\n            # offset_mapping — список пар; индексация `[0]` здесь была багом (len(int) падал на\n            # первом же вызове, найденном E-45b).\n            toks = tokenizer(\n                text, max_length=max_length, truncation=True,\n                add_special_tokens=False, return_offsets_mapping=True,\n            )\n            if len(toks["input_ids"]) >= min_length:\n                accepted.append(text)\n                samples.append({\n                    "input_ids": toks["input_ids"],\n                    "offset_mapping": torch.tensor(\n                        toks["offset_mapping"], dtype=torch.int32\n                    ),\n                })\n        else:\n            tokens = tokenizer(\n                text, max_length=max_length, truncation=True,\n                return_tensors="pt",\n            )\n            if tokens["input_ids"].shape[1] >= min_length:\n                accepted.append(text)\n                samples.append(tokens)\n        if len(samples) >= n_samples:\n            break\n\n    if texts is None:\n        texts = accepted\n        fp = pool_fingerprint(texts)\n        print(f"[calib] {domain}: поток с Hub, строк={len(texts)}, pool_fp={fp}", flush=True)\n        if os.environ.get(PIN_WRITE_ENV) == "1":\n            if pin is None:\n                raise RuntimeError(\n                    f"{PIN_WRITE_ENV}=1, но каталога пинов нет: {PIN_DIR_ENV} не задан, а "\n                    "`__file__` недоступен (flattening-ядро). Задайте PRISM_CALIB_PIN_DIR "\n                    "на путь, который попадает в вывод ядра."\n                )\n            pin.parent.mkdir(parents=True, exist_ok=True)\n            pin.write_text(\n                json.dumps(\n                    {\n                        "protocol": {"domain": domain, "model": model_name, "n_samples": n_samples,\n                                     "min_length": min_length, "max_length": max_length,\n                                     "offset_mapping": bool(save_offset_mapping)},\n                        "source": {"dataset": ds_id, "config": cfg, "split": split, "field": field},\n                        "fp": fp,\n                        "texts": texts,\n                    },\n                    ensure_ascii=False,\n                ),\n                encoding="utf-8",\n            )\n            print(f"[calib] {domain}: пин записан -> {pin}", flush=True)\n    else:\n        fp = pool_fingerprint(texts)\n        if len(samples) < n_samples:\n            raise RuntimeError(\n                f"пин {pin.name} дал {len(samples)} строк вместо {n_samples}: токенизатор или "\n                f"фильтр длины изменились относительно момента записи пина (pool_fp={fp}). "\n                f"Молча читать другой пул нельзя — числа стали бы невоспроизводимыми."\n            )\n    LAST_POOL_FPS[domain] = fp\n    return samples\n\n\ndef build_global_token_set(\n    model: AutoModelForCausalLM,\n    calibration_samples: list[dict[str, torch.Tensor]],\n    K: int = 64,\n) -> list[int]:\n    """Frequency token set: the K tokens that most often appear in the per-sample\n    top-K of the LAST position (`logits[:, -1]`) across the calibration pool.\n\n    One basis for every position, so a stored p-row is SUB-stochastic (~half of the\n    exp27b positions carry < 0.5 of the softmax mass) — that basis is what all\n    published Jacobian numbers sit on. The positional convention (each position gets\n    its own top-K, so p is a real truncation) is produced by\n    `experiments/exp27b_dense.py: column_mode="positional"` (cache exp27c), not here;\n    "union top-K at fixed K" is not definable, since the union of per-position top-K\n    exceeds K (audit4 Q10 verdict round). Never mix numbers from the two bases:\n    the basis belongs in the ChannelTag slice.\n    """\n    token_counter: Counter[int] = Counter()\n    model.eval()  # type: ignore[attr-defined]\n\n    device = next(model.parameters()).device  # type: ignore[attr-defined]\n    with torch.no_grad():\n        for sample in calibration_samples:\n            inputs = {\n                "input_ids": sample["input_ids"].to(device),\n                "attention_mask": sample["attention_mask"].to(device),\n            }\n            outputs = model(**inputs)  # type: ignore[operator]\n            logits = outputs.logits[:, -1, :]\n            top_idx = logits.topk(K, dim=-1).indices[0]\n            token_counter.update(top_idx.tolist())\n\n    U = [tid for tid, _ in token_counter.most_common(K)]\n    return sorted(U)\n\n\ndef load_models(\n    large_name: str, small_name: str, dtype: torch.dtype = torch.float32\n) -> tuple[torch.nn.Module, torch.nn.Module]:\n    """Load large and small models with eager attention.\n\n    dtype defaults to float32: the tested protocols run on CPU, where fp16\n    is ~10x slower (paper §12/C.9). Pass torch.float16 explicitly for GPU.\n    Thread count (torch.set_num_threads(4) in §12/C.9) is set by the caller\'s\n    setup, not here.\n    """\n    kwargs = {\n        "torch_dtype": dtype,\n        "attn_implementation": "eager",\n    }\n    large = AutoModelForCausalLM.from_pretrained(large_name, **kwargs)\n    large.eval()  # type: ignore[attr-defined]\n    small = AutoModelForCausalLM.from_pretrained(small_name, **kwargs)\n    small.eval()  # type: ignore[attr-defined]\n    return large, small\n\n\ndef extract_embeddings(\n    model: AutoModelForCausalLM,\n) -> torch.Tensor:\n    """Extract embedding matrix from model.\n\n    The bridge projects through the AN-embedding matrix W (§3/§5). For the\n    tied-weight checkpoints in scope, W = E (embed_tokens). If a checkpoint\n    is loaded WITHOUT tied weights, W = lm_head.weight ≠ E and the projection\n    would silently use the wrong matrix. We therefore warn ONCE per model if\n    lm_head.weight is not the same storage as embed_tokens.weight — a loud,\n    non-fatal signal that the W = E assumption does not hold for this model.\n    """\n    emb = model.model.embed_tokens.weight.detach()  # type: ignore[attr-defined]\n    _warn_if_not_tied(model, emb)\n    return emb\n\n\ndef _warn_if_not_tied(model: AutoModelForCausalLM, emb: torch.Tensor) -> None:\n    lm_head = getattr(getattr(model, "lm_head", None), "weight", None)\n    if lm_head is None:\n        return\n    if not lm_head.data_ptr() == emb.data_ptr():\n        # data_ptr identity covers tied weights; fall back to value equality\n        # for the (rare) case of a tied-but-copied view.\n        if not torch.equal(lm_head.detach(), emb):\n            warnings.warn(\n                f"extract_embeddings: lm_head.weight is NOT tied to "\n                f"embed_tokens.weight (model={type(model).__name__}). The "\n                f"bridge projects through E assuming W = E (§3); with untied "\n                f"weights the an-embedding matrix is lm_head.weight and the "\n                f"gap/bridge numbers are silently wrong. Extract lm_head "\n                f"explicitly if this is intended.",\n                UserWarning,\n                stacklevel=2,\n            )\n', encoding='utf-8')
print('[bundle] wrote 6 files under', ROOT)


In [ ]:
# ---------- driver ----------
# E-52k driver (DRIVER MODE, инлайн-бандл, без датасетов).
# Кривая τ(size) на GPU bf16: 0.8B → 2B → 4B (+повтор 0.8B как якорь протокола)
# и честный PTQ с HF-весов: bnb-int8 на 4B и 9B (7B в семье Qwen3.5 нет — 9B
# и есть «третий-крупный»; bf16-9B в T4 не влезает, int8 влезает).
import os, sys, time, glob, json, shutil, traceback, gc, subprocess
t0 = time.time()
os.environ.setdefault("OMP_NUM_THREADS", "3")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
CODE = "/kaggle/working/code"
sys.path.insert(0, CODE)
sys.path.insert(0, os.path.join(CODE, "experiments"))
try:
    subprocess.run(["pip", "install", "-q", "-U", "transformers", "bitsandbytes"], check=False)
    from huggingface_hub import snapshot_download
    import importlib.util, torch
    def load(mod, fn):
        spec = importlib.util.spec_from_file_location(mod, os.path.join(CODE, "experiments", fn))
        X = importlib.util.module_from_spec(spec)
        sys.modules[mod] = X
        spec.loader.exec_module(X)
        return X
    E47 = load("e47", "exp47b_gen_kinematics.py")
    # PRISM-каналы: jcache синтезируем из литерала U128 (бинарник не нужен)
    d08 = "/kaggle/working/m_Qwen3.5-0.8B"
    if not glob.glob(d08 + "/*.safetensors"):
        prev = glob.glob("/kaggle/input/**/m_Qwen3.5-0.8B", recursive=True)
        if prev:
            d08_src = prev[0]
        else:
            d08_src = snapshot_download("Qwen/Qwen3.5-0.8B", local_dir=d08)
    os.environ["PRISM_QWEN_DIR"] = d08 if glob.glob(d08 + "/*.safetensors") else d08_src
    jc = "/kaggle/working/U128_synth.pt"
    torch.save({"U128": list(E47.U128)}, jc)
    os.environ["PRISM_JCACHE"] = jc
    print("[env] QWEN_DIR:", os.environ["PRISM_QWEN_DIR"], flush=True)
    E52 = load("e52", "exp52a_perturb_axis.py")   # после env (load_model читает при вызове)
    JOBS = [  # (tag, hf_repo, dtype, bnb-int8)
        ("q08_bf16", "Qwen/Qwen3.5-0.8B", "bfloat16", False),
        ("q2_bf16",  "Qwen/Qwen3.5-2B",   "bfloat16", False),
        ("q4_bf16",  "Qwen/Qwen3.5-4B",   "bfloat16", False),
        ("q4_int8",  "Qwen/Qwen3.5-4B",   "bfloat16", True),
        ("q9_int8",  "Qwen/Qwen3.5-9B",   "bfloat16", True),
    ]
    for tag, repo, dt, q8 in JOBS:
        try:
            name = repo.split("/")[-1]
            md = f"/kaggle/working/m_{name}"
            if not glob.glob(md + "/*.safetensors"):
                prev = glob.glob(f"/kaggle/input/**/m_{name}", recursive=True)
                md = prev[0] if prev else snapshot_download(repo, local_dir=md)
            rd = f"/kaggle/working/r52k_{tag}"; os.makedirs(rd, exist_ok=True)
            print(f"\n##### {tag} ({repo}, {dt}{', bnb-int8' if q8 else ''})", flush=True)
            E47.main(model=md, n_docs=4, max_new=160, device="cuda:0", results_dir=rd,
                     do_verify_basis=False, dtype=dt, load_in_8bit=q8)
            sys.argv = ["e52a", "--model", "qwen", "--n-docs", "16", "--model-dir", md,
                        "--families", "rtn",
                        "--npz", rd + "/exp47b_gen_kinematics.npz",
                        "--docs-json", rd + "/exp47b_gen_kinematics.json",
                        "--out", f"/kaggle/working/exp52k_tau_{tag}.json"]
            E52.main()
            if name in ("Qwen3.5-2B", "Qwen3.5-9B") and md.startswith("/kaggle/working"):
                shutil.rmtree(md, ignore_errors=True)   # освобождаем диск под следующий размер
            gc.collect(); torch.cuda.empty_cache()
        except Exception:
            print(f"##### {tag} FAILED", flush=True)
            traceback.print_exc()
    print("\n===== SUMMARIES =====", flush=True)
    for f in sorted(glob.glob("/kaggle/working/exp52k_tau_*.json")):
        s = json.load(open(f))["summary"]
        print(f.split("exp52k_tau_")[1], "base:", s["base"]["tau"], flush=True)
except Exception:
    traceback.print_exc()
print("DONE %.1fs" % (time.time() - t0))
